<a href="https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/9_aprendizaje_maquina/_assets/05_sesgo_y_varianza.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab" width="300"></a>

# Notebook 5 · Sesgo y varianza

Acompaña [Sesgo y varianza: de dónde sale el error](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/separar-los-datos/sesgo-y-varianza/). Duración: ~35 min.

Ejecuta primero la celda de arranque. Recorre los bloques en orden: apuesta
antes de ejecutar «Juega», mueve un control y contrasta tu respuesta con
«Explicación». Puedes hacerlo por tu cuenta.

Ubica el punto de comparación (*benchmark*) en las salidas:
«la media de entrenamiento como punto de comparación; la clase 3 lo define».

In [ ]:
#@title ▶ Corre esta celda primero
import os, sys, glob, base64, zlib, importlib, urllib.request
URL = os.environ.get("ML_LAB_URL",
      "https://raw.githubusercontent.com/raya-lucaria/ia_o26/main/cou"
      "rse/9_aprendizaje_maquina/_assets/ml_lab.py")
REQUIERE = "1.6"                     # la versión con que se generó este notebook
_RESPALDO = "..."

def _v(texto): return tuple(int(p) for p in texto.split("."))

def _sirve(version):                 # misma X y una Y mayor o igual
    (x, y), (x0, y0) = _v(version), _v(REQUIERE)
    return x == x0 and y >= y0

def _cargar():                       # importa ml_lab DE CERO (sin la copia en memoria)
    sys.modules.pop("ml_lab", None)
    for viejo in glob.glob("__pycache__/ml_lab.*.pyc"):
        os.remove(viejo)             # un .pyc de la copia vieja taparía la nueva
    importlib.invalidate_caches()
    import ml_lab
    return ml_lab if _sirve(ml_lab.VERSION) else None

ml = None
if os.path.exists("ml_lab.py"):      # 1) la copia local, si ya es nueva
    try: ml = _cargar()
    except Exception: ml = None
if ml is None:                       # 2) la del repositorio
    try:
        urllib.request.urlretrieve(URL, "ml_lab.py"); ml = _cargar()
    except Exception: ml = None
if ml is None:                       # 3) el respaldo: sin red o repo aún sin esta versión
    with open("ml_lab.py", "wb") as archivo:
        archivo.write(zlib.decompress(base64.b64decode(_RESPALDO)))
    ml = _cargar()
ml.requiere(REQUIERE)
ml.tema("claro")
from ml_lab import *
print(f"Listo: ml_lab {ml.VERSION}")

Una recta puede equivocarse casi igual con cada muestra; una curva flexible
puede cambiar mucho entre muestras. Separaremos ese desplazamiento, esa
dispersión y el ruido de una entrega nueva. Con pérdida cuadrática:

$$E_{\mathcal D,\varepsilon}\bigl[(y-\widehat f(x))^2\mid x\bigr]=\sigma_\varepsilon^2(x)+\text{sesgo}(x)^2+\text{varianza}(x).$$

La identidad [Ruido, sesgo y varianza](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/separar-los-datos/sesgo-y-varianza/#raya-object-ml-c2-teo-sesgo-varianza)
promedia sobre entrenamientos y ruido nuevo, independiente del entrenamiento.
Supone segundos momentos finitos, todas las variables visibles y
$E[\varepsilon\mid x]=0$. Aquí $\varepsilon$ es ruido; recuerda los avisos de
letras del curso. En el proceso inicial, su aportación es $9.0$.

**Predice.** ¿Qué grado tendrá el abanico más ancho en $x_0=5$: 1, 4 o 12?

Apuesta un número: el grado. Conserva «x0 (km)» en 5 y compara «grado N».
Activa «ver la diana». Aquí $x_0$ es un punto fijo de $x$, no el centro
de Taylor de la clase 1.

In [ ]:
#@title Juega: el abanico y la diana
import functools, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEMILLA = 0
construir = lambda N: (lambda df, N=N: ml.ajustar(df, phi=ml.polinomio(N)))
CANDIDATOS = {N: construir(N) for N in range(13)}


def fabrica(s, n=60, sigma=3.0):
    return ml.PROCESOS["entregas_curvas"](n=n, semilla=int(s), sigma_e=sigma,
                                         n_realidad=2000)


SV = ml.sesgo_varianza(fabrica, CANDIDATOS, repeticiones=100, semilla=SEMILLA)
X_MALLA = SV.malla["x"].to_numpy()


def bloque_1(N, x0, diana):
    i = int(np.argmin(np.abs(X_MALLA - x0)))      # el punto de la malla más cercano
    p = SV.predicciones[N][:, i]
    if diana:
        ml.dibujar_diana(SV, x0, candidatos=[N])
        margen = 0.05 * (p.max() - p.min()) + 2    # ninguna marca queda fuera del eje
        plt.xlim(min(15, p.min() - margen), max(55, p.max() + margen))
    else:
        ml.dibujar_abanico(SV, N)
        plt.ylim(0, 50)
        fuera = int((np.abs(SV.predicciones[N] - 25) > 25).any(axis=1).sum())
        if fuera:
            print(f"{fuera} de {len(p)} curvas salen del eje de 0 a 50 en algún punto")
    plt.show()
    print(f"grado {N}, x0 = {X_MALLA[i]:.2f} km: E[y|x0] = {SV.verdad[i]:.1f};"
          f" sesgo = {p.mean() - SV.verdad[i]:.2f}; varianza = {p.var():.2f}"
          f" (desviación {p.std():.2f})")


ml.laboratorio(bloque_1, N=ml.deslizador("grado N", 0, 12, 1, 4),
               x0=ml.deslizador("x0 (km)", 0.5, 5.5, 0.25, 5.0),
               diana=ml.interruptor("ver la diana", False))

**Mira.** Cada línea delgada viene de otra muestra; la gruesa discontinua
promedia ajustes; la verde muestra $E[y\mid x]$. En la diana, cada marca
es una predicción; las dos líneas indican promedio y verdad.
El control evita los bordes: el grado 12 puede extrapolar allí y perder
escala. Consulta sesgo y varianza impresos al mover «x0 (km)».
El eje de la diana se estira para mostrar todas las predicciones; con el grado 12 cerca de un borde puede estirarse mucho.
En el abanico, el notebook imprime cuántas curvas salen del eje de 0 a 50.

<details><summary>Explicación</summary>

Con los valores iniciales, $E[y\mid x_0]=31.5$. Al comparar los grados en
ese punto, gana el 12 en anchura: desviación 3.5, frente a 0.9 del grado 1.
Sus sesgos son $-0.5$ y $1.5$, respectivamente, con 100 muestras.

En [La diana en x = 5](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/separar-los-datos/sesgo-y-varianza/#raya-object-ml-c2-diana),
$\text{sesgo}(x)=E_{\mathcal D}[\widehat f(x)]-f(x)$ mide el desplazamiento;
$\text{varianza}(x)=\operatorname{Var}_{\mathcal D}(\widehat f(x))$ mide la dispersión.
Aquí $f(x)=E[y\mid x]$.

Entre las aportaciones del ajuste, domina el sesgo² en el grado 1 y la varianza en el grado 12; en el error total del grado 1 domina el ruido, 9.0.
La recta se equivoca parecido; el grado alto cambia más entre muestras.
Es una comparación local: verifica las cifras al cambiar $x_0$.

</details>

**Predice.** Si duplicas el ruido $\sigma_\varepsilon$, ¿qué barra crece: el ruido, el sesgo², la varianza o varias?

Apuesta un número: el factor por el que cambiará la aportación del ruido.
En «ruido σε», pasa de 3 a 6; deja «entregas n» en 60.
La etiqueta del control nombra $\sigma_\varepsilon$, la desviación del ruido.

In [ ]:
#@title Juega: la descomposición, numérica
SIGMAS, NS = (1.5, 3.0, 6.0), (30, 60, 120)
@functools.lru_cache(maxsize=None)          # cada combinación, la primera vez
def sv_b2(s, n):
    return ml.sesgo_varianza(lambda r: fabrica(r, n=n, sigma=s), CANDIDATOS,
                             repeticiones=50, semilla=SEMILLA)


def bloque_2(sigma, n):
    sv = sv_b2(sigma, n)
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4))
    ml.dibujar_sesgo_varianza(sv, ax=a)
    datos = fabrica(SEMILLA, n=n, sigma=sigma)
    s = ml.elegir(datos, CANDIDATOS, con="cv", k=5)
    ml.dibujar_cv(s, ax=b)
    b.set_yscale("log")
    plt.show()
    t = sv.tabla.set_index("candidato")
    N0 = int(t["total"].idxmin())
    bench = t.loc[0, "total"]                 # N = 0: error esperado de la media de ent
    print(f"σε = {sigma:g}, n = {n}: menor total en N⋆ = {N0}"
          f" (ruido {t.loc[N0, 'ruido']:.2f},"
          f" sesgo² {t.loc[N0, 'sesgo2']:.2f}, varianza {t.loc[N0, 'varianza']:.2f})")
    print(f"benchmark (N = 0, la media de entrenamiento): error esperado {bench:.2f};"
          f" la CV_5 de una muestra elige N̂ = {s.elegido}")


ml.laboratorio(bloque_2, sigma=ml.selector("ruido σε", list(SIGMAS), 3.0),
               n=ml.selector("entregas n", list(NS), 60))

**Mira.** A la izquierda, cada barra apila ruido, sesgo² y varianza de un
grado; la estrella marca el menor total. Algunas barras salen del eje.
A la derecha, cada punto resume validación cruzada (*cross-validation*)
con cinco pliegues; el eje es logarítmico. Compara ambos mínimos y el
benchmark impreso. En $\mathrm{CV}_k$, $k$ cuenta pliegues: recuerda el aviso de letras.

<details><summary>Explicación</summary>

Con los valores iniciales, el menor total calculado está en grado 2;
la CV de una muestra elige 6. El ruido aporta 9.0.

La identidad de [Sesgo y varianza: de dónde sale el error](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/separar-los-datos/sesgo-y-varianza/#raya-object-ml-c2-teo-sesgo-varianza)
suma $\sigma_\varepsilon^2(x)+\text{sesgo}(x)^2+\text{varianza}(x)$.
Duplicar $\sigma_\varepsilon$ cuadruplica el ruido y también aumenta la
varianza del ajuste. En mínimos cuadrados, con entradas fijas y ruido
centrado, no cambia el ajuste esperado; las barras estimadas sí fluctúan.

Más ruido tiende a favorecer grados menores; más datos, mayores.
Consulta el mínimo impreso de cada combinación. La estrella corresponde al
óptimo teórico, aproximado repitiendo muestras; el gorro corresponde a la
elección con datos. Una muestra puede discrepar del promedio.

</details>

**Predice.** Al pasar de 30 a 960 entregas, ¿cuál baja más: el grado 1 o el 10?

Apuesta un número: cuánto esperas que baje el riesgo de la recta.
Puedes elegir el grado 1, el 10 o una caída parecida.
Compara «1 y 10» en «modelos»; después activa «panel del grado elegido».

In [ ]:
#@title Juega: más datos
NS_B3 = [30, 60, 120, 240, 480, 960]
fabrica_960 = lambda s: ml.PROCESOS["entregas_curvas"](  # una realidad por muestra
    n=960, semilla=int(s), n_realidad=2000, semilla_realidad=int(s),
    fracciones=(1, 0, 0))
CURVA = ml.curva_aprendizaje(fabrica_960, {N: construir(N) for N in (1, 4, 10)},
                             ns=NS_B3, repeticiones=50, semilla=SEMILLA)
ELEGIDO = ml.capacidad_elegida(fabrica_960, CANDIDATOS, ns=NS_B3, k=5, repeticiones=20,
                               semilla=SEMILLA)
PISO = ml.piso(fabrica_960(SEMILLA))
_D960 = fabrica_960(SEMILLA)
BENCH_B3 = ml.benchmarks(_D960, _D960.despliegue).loc["media"].iloc[0]  # media de ent


def bloque_3(modelos, abajo):
    grados = [1, 10] if modelos == "1 y 10" else [1, 4, 10]
    t = CURVA[CURVA["candidato"].isin(grados)]
    ml.dibujar_curva_aprendizaje(t, piso=PISO, elegido=ELEGIDO if abajo else None,
                                 resumen="mediana")
    plt.show()
    med = t.groupby(["candidato", "n"])["R"].median()
    for N in grados:
        print(f"grado {N}: R (mediana de 50) con 30 = {med[N, 30]:.1f}, con 960 ="
              f" {med[N, 960]:.1f}; piso {PISO:.1f}")
    e = ELEGIDO.groupby("n")["elegido"].median()
    print(f"grado que elige la CV_5 (mediana de 20): {e[30]:g} con 30,"
          f" {e[960]:g} con 960")
    print(f"benchmark (la media de entrenamiento, 960 entregas) en el despliegue ="
          f" {BENCH_B3:.1f}")


ml.laboratorio(bloque_3, modelos=ml.selector("modelos", ["1 y 10", "1, 4 y 10"]),
               abajo=ml.interruptor("panel del grado elegido", True))

**Mira.** Los círculos muestran $L_{\text{ent}}$, pérdida promedio en entrenamiento;
las líneas sin círculos, $R$ en el despliegue. Son medianas de 50 muestras por
tamaño. La punteada marca el piso cuadrático.
Abajo, cada cuadrado resume el grado elegido por CV: mediana de 20 muestras.
Los estilos distinguen grados.
La última línea impresa da el benchmark: la media de entrenamiento como punto de
comparación (la clase 3 lo define), evaluada en el despliegue.

<details><summary>Explicación</summary>

Con los valores iniciales, baja más el grado 10, que empieza arriba de la
recta. Las medianas de $R$ del grado 1 son 15.6, 14.9 y 14.9 con
30, 480 y 960 entregas. Las del grado 10 son 9.2 y 9.1 con 480 y 960;
el piso es 9.0.

En general, más datos pueden reducir la varianza sin corregir una familia
inadecuada. [Ruido, sesgo y varianza](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/separar-los-datos/sesgo-y-varianza/#raya-object-ml-c2-teo-sesgo-varianza)
da $E_{\mathcal D}[R(\widehat f)]=R_{\text{Bayes}}+E_x[\text{sesgo}(x)^2]+E_x[\text{varianza}(x)]$.
La identidad habla de medias; aquí miras medianas.

La recta conserva error de aproximación. La CV tiende a elegir más capacidad:
su mediana pasa de 3 a 10. En una repetición puede bajar.

</details>

**Predice.** Proceso misterioso A: ¿sobreajuste, no aprende, subajuste o listo?

Apuesta un número: cuánto esperas reducir la CV al subir el grado.
Repite la pregunta para B, C y D en «proceso».
Sobreajuste (*overfitting*) y subajuste (*underfitting*) son dos diagnósticos posibles.
Antes de revelar el piso, usa las curvas y la comparación con el benchmark.

In [ ]:
#@title Juega: cuatro procesos misteriosos
PROCESOS_B4 = {   # (datos, grado del modelo actual); los de ml_cifras_c2
    "A": (ml.PROCESOS["entregas_curvas"](n=400, semilla=SEMILLA, n_realidad=2000), 1),
    "B": (ml.PROCESOS["entregas_curvas"](n=40, semilla=SEMILLA, n_realidad=2000), 12),
    "C": (ml.proceso(n=400, x={"x": ml.uniforme(0, 6)}, f=lambda d: 30 + 0.2 * d["x"],
                     ruido=ml.normal(0, 8), semilla=SEMILLA, n_realidad=2000), 1),
    "D": (ml.PROCESOS["entregas_curvas"](n=400, semilla=SEMILLA, n_realidad=2000), 4)}


def bloque_4(proceso, revelar):
    datos, N = PROCESOS_B4[proceso]
    grados = range(max(8, N) + 1)
    tabla = ml.curva_capacidad(datos, {g: construir(g) for g in grados},
                               conjuntos=("ent", "cv"))
    cv = {g: ml.validacion_cruzada(datos, construir(g), k=5) for g in (0, N)}
    ml.dibujar_curva_capacidad(tabla, marcar=N, benchmark=(cv[0].media, cv[0].ee),
                               piso=ml.piso(datos) if revelar else None)
    plt.yscale("log"); plt.show()
    modelo, bench = construir(N)(datos.ent), construir(0)(datos.ent)   # solo con ent
    y = datos.val["y"].to_numpy()
    d, ee = ml.diferencia_pareada(ml.perdida(y, modelo.predecir(datos.val)),
                                  ml.perdida(y, bench.predecir(datos.val)))
    gana = "sí" if d < -2 * ee else "no"
    print("(a) en validación (la validación no eligió este modelo): modelo − benchmark"
          f" = {d:.2f} (EE {ee:.2f}); le gana por más de dos EE: {gana}")
    dp = ml.diferencia_por_pliegue(cv[N], cv[0])
    por, media = dp["diferencia"].to_numpy(float)[:5], float(dp["diferencia"].iloc[5])
    senal = "señal" if abs(media) > por.std(ddof=1) else "falta evidencia"
    print(f"(b) con CV (descriptivo): diferencia media {media:.2f}; por pliegue "
          + ", ".join(f"{v:.2f}" for v in por) + f" → {senal}")
    print(f"    {dp.attrs['nota']}")
    if revelar:   # con la media condicional conocida: R − piso no puede ser negativo
        p, D = ml.piso(datos), datos.despliegue
        exceso = np.mean((modelo.predecir(D) - np.asarray(datos.f_visible(D))) ** 2)
        print(f"solo en el simulador: R = {p + exceso:.2f}, piso {p:.2f},"
              f" por ganar {exceso:.2f}")


ml.laboratorio(bloque_4, proceso=ml.selector("proceso", list(PROCESOS_B4)),
               revelar=ml.interruptor("revelar el piso y R (solo en el simulador)"))

**Mira.** Cada círculo mide entrenamiento; cada triángulo, CV por grado;
la raya-punto marca el benchmark. Lee las dos líneas: (a) decide la
comparación pareada en validación; (b) da una señal descriptiva de CV.
Activa «revelar el piso y R (solo en el simulador)»: `datos.despliegue`
es el visor de la realidad; solo existe en el simulador.

<details><summary>Explicación</summary>

Con los valores iniciales, A muestra subajuste: CV $15.1\pm0.9$ frente a 9.7
con grado 8. B muestra sobreajuste: entrenamiento 5.7 frente a CV 262.2.
C no aprende: diferencia pareada $-0.1$ (EE 0.3). D está listo, con CV 9.7.

En general, bajar CV más de un EE es heurística. En (a), ambos modelos
están fijados; la validación no los eligió. Usa el EE de la diferencia
pareada de pérdidas para la regla de dos EE. En (b), A y D dan «señal»;
C, «falta evidencia»: las pérdidas fuera de pliegue dependen entre sí.

Al revelar, lee $R-R_{\text{Bayes}}$, incluso en D.
[Ruido, sesgo y varianza](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/separar-los-datos/sesgo-y-varianza/#raya-object-ml-c2-teo-sesgo-varianza)
da $E_{\mathcal D}[R(\widehat f)]-R_{\text{Bayes}}=E_x[\text{sesgo}(x)^2]+E_x[\text{varianza}(x)]$.

</details>

**Taller.** Ejecuta el ejemplo activo. Después copia una alternativa comentada
y quita sus signos `#`; conserva la llamada final a
`ml.taller(datos, clase=2, que="descomponer")`.

Otra $f$ cambia la forma que intentas aprender. Más ruido cambia
`sigma_e`: observa el piso y la dispersión. Con $n=200$, explora qué grados
aprovechan más datos. Cambia una alternativa por vez; la última asignación
a `datos` determina el proceso.

Antes de ejecutar, predice qué componente cambiará. Compara las barras con
el caso inicial y explica si necesitas otra familia, más datos o mejores
variables.

El taller promedia sobre todo el rango de $x$, bordes incluidos; con los datos iniciales imprime para el grado 12 un sesgo² de 603 y una varianza de decenas de miles.
No significa que el grado 12 esté sesgado en el centro: en los bordes, unos pocos de los 50 ajustes se disparan y arrastran el promedio, por lo que ese sesgo² estimado es inestable.
Léelo como señal de extrapolación, junto a la varianza.

In [ ]:
datos = ml.PROCESOS["entregas_curvas"](n=60, semilla=0)
# 1) otra f:      datos = ml.proceso(n=60, x={"x": ml.uniforme(0, 6)},
#                                    f=lambda d: 10 + 12 / (1 + d.x))
# 2) más ruido:   datos = ml.PROCESOS["entregas_curvas"](n=60, semilla=0, sigma_e=6)
# 3) n = 200:     datos = ml.PROCESOS["entregas_curvas"](n=200, semilla=0)
ml.taller(datos, clase=2, que="descomponer")

**Galería.** Los casos 13–16 son misterios: abre cada caso, lee su pregunta
y escribe tu diagnóstico antes de revelar «¿Qué se ve?».
Busca qué comparación sostiene tu respuesta y qué dato te haría cambiarla.

En el caso 13, compara $L_{\text{prueba}}$ con $R$: el polinomio usa solo la distancia, sin tiempo, y la prueba al azar casi no engaña.
Contrasta lo que ves con la explicación: la página muestra el engaño con vecinos en distancia y tiempo.
Toma el caso como un aviso para revisar cómo separas los datos.

In [ ]:
#@title Galería: casos que se rompen a la vista
ml.galeria(clase=2)

**Ejercicio.** En un $x$ fijo, $f(x)=11$. Un procedimiento predice $8$, $10$ o $12$,
cada una con probabilidad $1/3$ según la muestra, y el ruido nuevo tiene varianza 4.
Calcula sesgo, varianza y error cuadrático esperado.

**Pista:** ¿Alrededor de qué número se mide la varianza: de $f(x)$ o del promedio de las predicciones?

<details><summary>Explicación</summary>

El promedio es 10. El sesgo es $-1$: el procedimiento predice, en promedio,
por debajo de $f(x)=11$.
La varianza mide las desviaciones cuadráticas respecto de 10:

$$\frac{4+0+4}{3}=\frac83.$$

Por [Ruido, sesgo y varianza](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/separar-los-datos/sesgo-y-varianza/#raya-object-ml-c2-teo-sesgo-varianza),
el error cuadrático esperado es

$$4+1+\frac83=\frac{23}{3}.$$

El sesgo puede ser negativo; su contribución al error es su cuadrado.
El ruido nuevo se suma aunque el ajuste promedio acierte.

</details>

Repetir muestras separa desplazamiento y dispersión. Con 960 entregas,
la mediana del riesgo del grado 10 llega a 9.1; la recta conserva 14.9.
Más datos ayudan de manera distinta según la familia. Una curva aislada
no cuenta toda la historia.

¿Cómo bajo la varianza sin bajar el grado? (clase 3).